# Road Accidents France 2023 - Modeling

Multi-class classification of `grav` (1=unscathed, 2=killed, 3=hospitalised, 4=light) at user-level grain.
Joins 4 BAAC tables on `Num_Acc` + `id_vehicule`/`num_veh`. Trains Logistic Regression, Random Forest, XGBoost.
Outputs metrics + best model to `deliverables/`.

In [1]:
import json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (balanced_accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
DELIV = ROOT / 'deliverables'
DELIV.mkdir(exist_ok=True)
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
RANDOM_STATE = 42
print('paths ok')

paths ok


## 1. Load and join the 4 BAAC tables

In [2]:
def read_baac(name):
    fp = DATA / f'{name}-2023.csv'
    try:
        return pd.read_csv(fp, sep=';', encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(fp, sep=';', encoding='latin-1', low_memory=False)

caract = read_baac('caract')
lieux = read_baac('lieux')
usagers = read_baac('usagers')
vehicules = read_baac('vehicules')
for n, d in [('caract', caract), ('lieux', lieux), ('usagers', usagers), ('vehicules', vehicules)]:
    print(n, d.shape)

caract (54822, 15)
lieux (70860, 18)
usagers (125789, 16)
vehicules (93585, 11)


In [3]:
# Drop unknown grav (-1) early
usagers = usagers[usagers['grav'] != -1].copy()

# Deduplicate lieux to one row per accident (keep first segment)
lieux_uniq = lieux.drop_duplicates(subset='Num_Acc', keep='first')

# Join: usagers + vehicules on (Num_Acc, id_vehicule, num_veh) ; then caract, then lieux on Num_Acc
df = usagers.merge(vehicules, on=['Num_Acc', 'id_vehicule', 'num_veh'], how='left', suffixes=('', '_veh'))
df = df.merge(caract, on='Num_Acc', how='left', suffixes=('', '_car'))
df = df.merge(lieux_uniq, on='Num_Acc', how='left', suffixes=('', '_lieu'))
print('joined shape:', df.shape)
print('grav distribution:')
print(df['grav'].value_counts().sort_index())

joined shape: (125671, 55)
grav distribution:
grav
1    53399
2     3398
3    19271
4    49603
Name: count, dtype: int64


## 2. Feature engineering

- hour from `hrmn` (HH:MM string)
- weekday from (`an`, `mois`, `jour`)
- age = 2023 - `an_nais`
- replace BAAC sentinel `-1` with NaN on numeric features
- vehicle category, road category, speed limit kept as-is

In [4]:
# hour
df['hour'] = df['hrmn'].astype(str).str.split(':').str[0].astype(float)

# weekday (0=Monday)
df['date'] = pd.to_datetime(dict(year=df['an'], month=df['mois'], day=df['jour']), errors='coerce')
df['weekday'] = df['date'].dt.weekday

# age
df['age'] = 2023 - pd.to_numeric(df['an_nais'], errors='coerce')
df.loc[(df['age'] < 0) | (df['age'] > 110), 'age'] = np.nan

# Feature columns
num_features = ['hour', 'weekday', 'age', 'vma']
cat_features = ['catu', 'sexe', 'trajet', 'secu1', 'place',
                'catv', 'obs', 'obsm', 'choc', 'manv', 'motor',
                'lum', 'agg', 'int', 'atm', 'col',
                'catr', 'circ', 'nbv', 'surf', 'infra', 'situ', 'prof', 'plan']

# Sentinel -> NaN (BAAC encodes missing as -1)
for c in num_features + cat_features:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df.loc[df[c] == -1, c] = np.nan

# Drop rows missing the target (already filtered) and require core features
df = df.dropna(subset=['grav'])
X = df[num_features + cat_features].copy()
y = df['grav'].astype(int).copy()

print('X:', X.shape, 'y:', y.shape)
print('NaN per col (top 10):')
print(X.isna().sum().sort_values(ascending=False).head(10))

X: (125671, 28) y: (125671,)
NaN per col (top 10):
circ      8602
vma       4438
nbv       3881
age       2480
trajet    2385
sexe      2315
secu1     2184
infra     1081
motor      218
situ       117
dtype: int64


## 3. Train / val / test split (stratified, 70 / 15 / 15)

In [5]:
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15 / 0.85, stratify=y_tv, random_state=RANDOM_STATE)
print('train', X_train.shape, 'val', X_val.shape, 'test', X_test.shape)
print('train class dist:')
print(y_train.value_counts(normalize=True).round(3).sort_index())

train (87969, 28) val (18851, 28) test (18851, 28)
train class dist:
grav
1    0.425
2    0.027
3    0.153
4    0.395
Name: proportion, dtype: float64


## 4. Helper: evaluate

In [6]:
GRAV_LABELS = {1: 'unscathed', 2: 'killed', 3: 'hospitalised', 4: 'light'}

def evaluate(name, model, X_eval, y_eval, label_encoder=None):
    y_pred = model.predict(X_eval)
    if label_encoder is not None:
        y_pred = label_encoder[y_pred]
    bal = balanced_accuracy_score(y_eval, y_pred)
    macro = f1_score(y_eval, y_pred, average='macro')
    print(f'\n=== {name} ===')
    print(f'balanced_accuracy: {bal:.4f}')
    print(f'macro_f1:          {macro:.4f}')
    print(classification_report(y_eval, y_pred, target_names=[GRAV_LABELS[c] for c in sorted(y_eval.unique())], digits=3))
    cm = confusion_matrix(y_eval, y_pred, labels=sorted(y_eval.unique()))
    print('confusion matrix (rows=true, cols=pred):')
    print(pd.DataFrame(cm, index=[GRAV_LABELS[c] for c in sorted(y_eval.unique())],
                       columns=[GRAV_LABELS[c] for c in sorted(y_eval.unique())]))
    rep = classification_report(y_eval, y_pred, output_dict=True)
    return {
        'name': name,
        'balanced_accuracy': float(bal),
        'macro_f1': float(macro),
        'per_class': {GRAV_LABELS[int(k)]: v for k, v in rep.items() if k.isdigit()},
        'confusion_matrix': cm.tolist(),
    }

## 5. Baseline 1: Logistic Regression (numeric scaled, categorical one-hot via label-as-int + impute)

In [7]:
preproc = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_features),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_features),
])

logreg = Pipeline([
    ('pre', preproc),
    ('clf', LogisticRegression(max_iter=500, class_weight='balanced',
                                solver='lbfgs', n_jobs=4, random_state=RANDOM_STATE))
])
logreg.fit(X_train, y_train)
metrics_lr = evaluate('LogisticRegression (val)', logreg, X_val, y_val)


=== LogisticRegression (val) ===
balanced_accuracy: 0.4637
macro_f1:          0.3872
              precision    recall  f1-score   support

   unscathed      0.658     0.677     0.668      8010
      killed      0.084     0.547     0.145       510
hospitalised      0.268     0.224     0.244      2890
       light      0.622     0.406     0.492      7441

    accuracy                          0.497     18851
   macro avg      0.408     0.464     0.387     18851
weighted avg      0.569     0.497     0.519     18851

confusion matrix (rows=true, cols=pred):
              unscathed  killed  hospitalised  light
unscathed          5424     910           633   1043
killed               63     279            92     76
hospitalised        520    1005           648    717
light              2231    1144          1042   3024


## 6. Baseline 2: Random Forest (handles NaN via simple imputation; default params)

In [8]:
rf_pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', SimpleImputer(strategy='most_frequent'), cat_features),
])
rf = Pipeline([
    ('pre', rf_pre),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=4,
                                    class_weight='balanced_subsample',
                                    random_state=RANDOM_STATE))
])
rf.fit(X_train, y_train)
metrics_rf = evaluate('RandomForest (val)', rf, X_val, y_val)


=== RandomForest (val) ===
balanced_accuracy: 0.4745
macro_f1:          0.4880
              precision    recall  f1-score   support

   unscathed      0.731     0.840     0.782      8010
      killed      0.438     0.076     0.130       510
hospitalised      0.502     0.327     0.396      2890
       light      0.634     0.654     0.644      7441

    accuracy                          0.667     18851
   macro avg      0.576     0.474     0.488     18851
weighted avg      0.650     0.667     0.651     18851

confusion matrix (rows=true, cols=pred):
              unscathed  killed  hospitalised  light
unscathed          6732       3           178   1097
killed               59      39           228    184
hospitalised        386      36           946   1522
light              2032      11           534   4864


## 7. Improved model: XGBoost with class weighting + small randomised hyper-param search

XGBoost expects 0-indexed class labels. We map `grav` -> {0,1,2,3} and back.

In [9]:
classes_sorted = np.array(sorted(y_train.unique()))  # [1,2,3,4]
to_xgb = {c: i for i, c in enumerate(classes_sorted)}
from_xgb = classes_sorted  # index -> original label

y_train_x = y_train.map(to_xgb).values
y_val_x = y_val.map(to_xgb).values
y_test_x = y_test.map(to_xgb).values

# sample weights from inverse class freq
sw_train = compute_sample_weight(class_weight='balanced', y=y_train_x)

# Pre-impute numerically (XGBoost handles NaN natively, but we keep a uniform pipeline for cat too)
imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')
X_train_num = pd.DataFrame(imputer_num.fit_transform(X_train[num_features]), columns=num_features, index=X_train.index)
X_val_num = pd.DataFrame(imputer_num.transform(X_val[num_features]), columns=num_features, index=X_val.index)
X_test_num = pd.DataFrame(imputer_num.transform(X_test[num_features]), columns=num_features, index=X_test.index)
X_train_cat = pd.DataFrame(imputer_cat.fit_transform(X_train[cat_features]), columns=cat_features, index=X_train.index)
X_val_cat = pd.DataFrame(imputer_cat.transform(X_val[cat_features]), columns=cat_features, index=X_val.index)
X_test_cat = pd.DataFrame(imputer_cat.transform(X_test[cat_features]), columns=cat_features, index=X_test.index)
X_train_xgb = pd.concat([X_train_num, X_train_cat], axis=1).astype(float)
X_val_xgb = pd.concat([X_val_num, X_val_cat], axis=1).astype(float)
X_test_xgb = pd.concat([X_test_num, X_test_cat], axis=1).astype(float)
print('xgb input shape:', X_train_xgb.shape)

xgb input shape: (87969, 28)


In [10]:
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.15],
    'n_estimators': [200, 400],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 5],
}

base_xgb = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4, eval_metric='mlogloss',
    tree_method='hist', n_jobs=4, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    base_xgb, param_distributions=param_grid, n_iter=5, cv=3,
    scoring='f1_macro', n_jobs=1, random_state=RANDOM_STATE, verbose=0)
search.fit(X_train_xgb, y_train_x, sample_weight=sw_train)
print('best params:', search.best_params_)
print('best CV macro_f1:', round(search.best_score_, 4))
best_xgb = search.best_estimator_

best params: {'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
best CV macro_f1: 0.5664


In [11]:
metrics_xgb_val = evaluate('XGBoost tuned (val)', best_xgb, X_val_xgb,
                            pd.Series(y_val.values, index=y_val.index),
                            label_encoder=from_xgb)
metrics_xgb_test = evaluate('XGBoost tuned (test)', best_xgb, X_test_xgb,
                             pd.Series(y_test.values, index=y_test.index),
                             label_encoder=from_xgb)


=== XGBoost tuned (val) ===
balanced_accuracy: 0.5682
macro_f1:          0.4924
              precision    recall  f1-score   support

   unscathed      0.743     0.799     0.770      8010
      killed      0.144     0.575     0.231       510
hospitalised      0.390     0.431     0.409      2890
       light      0.695     0.468     0.559      7441

    accuracy                          0.606     18851
   macro avg      0.493     0.568     0.492     18851
weighted avg      0.654     0.606     0.617     18851

confusion matrix (rows=true, cols=pred):
              unscathed  killed  hospitalised  light
unscathed          6403     297           430    880
killed               33     293           142     42
hospitalised        272     766          1245    607
light              1908     674          1377   3482



=== XGBoost tuned (test) ===
balanced_accuracy: 0.5748
macro_f1:          0.4984
              precision    recall  f1-score   support

   unscathed      0.752     0.794     0.772      8010
      killed      0.151     0.578     0.240       510
hospitalised      0.394     0.456     0.422      2891
       light      0.688     0.471     0.559      7440

    accuracy                          0.609     18851
   macro avg      0.496     0.575     0.498     18851
weighted avg      0.655     0.609     0.620     18851

confusion matrix (rows=true, cols=pred):
              unscathed  killed  hospitalised  light
unscathed          6362     295           404    949
killed               36     295           131     48
hospitalised        243     739          1317    592
light              1822     621          1494   3503


## 8. Binary reframe: severe (grav in {2,3}) vs not-severe, AUROC

In [12]:
y_train_bin = y_train.isin([2, 3]).astype(int).values
y_val_bin = y_val.isin([2, 3]).astype(int).values
y_test_bin = y_test.isin([2, 3]).astype(int).values

scale_pos = (y_train_bin == 0).sum() / max((y_train_bin == 1).sum(), 1)
xgb_bin = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='auc', tree_method='hist',
    n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.9,
    colsample_bytree=0.9, scale_pos_weight=scale_pos,
    n_jobs=4, random_state=RANDOM_STATE)
xgb_bin.fit(X_train_xgb, y_train_bin)
p_val = xgb_bin.predict_proba(X_val_xgb)[:, 1]
p_test = xgb_bin.predict_proba(X_test_xgb)[:, 1]
auc_val = roc_auc_score(y_val_bin, p_val)
auc_test = roc_auc_score(y_test_bin, p_test)
print(f'binary severe-vs-not AUROC val: {auc_val:.4f}, test: {auc_test:.4f}')

binary severe-vs-not AUROC val: 0.8669, test: 0.8732


## 9. Persist best model + metrics

In [13]:
with open(DELIV / 'road_xgb.pkl', 'wb') as fh:
    pickle.dump({
        'model': best_xgb,
        'imputer_num': imputer_num,
        'imputer_cat': imputer_cat,
        'num_features': num_features,
        'cat_features': cat_features,
        'class_mapping': {int(k): int(v) for k, v in to_xgb.items()},
        'inverse_class_mapping': {int(i): int(c) for i, c in enumerate(from_xgb)},
    }, fh)

metrics_payload = {
    'baseline_logreg_val': metrics_lr,
    'baseline_rf_val': metrics_rf,
    'improved_xgb_val': metrics_xgb_val,
    'improved_xgb_test': metrics_xgb_test,
    'binary_severe_auroc': {'val': float(auc_val), 'test': float(auc_test)},
    'best_xgb_params': search.best_params_,
    'n_train': int(len(y_train)), 'n_val': int(len(y_val)), 'n_test': int(len(y_test)),
}
with open(DELIV / 'metrics.json', 'w') as fh:
    json.dump(metrics_payload, fh, indent=2, default=str)
print('saved:', DELIV / 'road_xgb.pkl')
print('saved:', DELIV / 'metrics.json')

saved: deliverables//road_xgb.pkl
saved: deliverables//metrics.json
